# vPCF Model Training on Google Colab

This notebook trains DEC/IDEC clustering models on vPCF data from HDF5 and DM3 files.

## Step 1: Setup Environment

In [6]:
# Check if running on Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running on Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

import sys
import os

Running on Google Colab


In [7]:
# Install required packages
!pip install -q h5py hyperspy tqdm scikit-learn pandas tensorflow

In [22]:
# Detect environment and setup paths
if IN_COLAB:
    # Check if we're in web Colab or VS Code Colab
    try:
        import google.colab.notebook
        IS_WEB_COLAB = True
        print("Running on Web Colab")
    except:
        IS_WEB_COLAB = False
        print("Running on VS Code Colab Extension")
    
    # Try to mount Google Drive for both environments
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("Google Drive mounted successfully at /content/drive")
        
        project_dir = '/content/drive/My Drive/CMU vPCF Project'
        os.chdir(project_dir)
        print(f"Changed to: {os.getcwd()}")
        
        h5_file = '/content/drive/My Drive/CMU vPCF Project/Experimentally-obtained vPCF Testing/data/vPCF_test_2.h5'
        dm3_file = '/content/drive/My Drive/CMU vPCF Project/Experimentally-obtained vPCF Testing/data/vPCF_test_2.dm3'
    except Exception as e:
        print(f"Drive mounting failed: {e}")
        print("Attempting to use local paths...")
        
        # Fallback for VS Code Colab without mounted drive
        project_dir = '/content'
        h5_file = '/content/vPCF_test_2.h5'
        dm3_file = '/content/vPCF_test_2.dm3'
        
        print(f"Using fallback paths")
        print(f"H5 file: {h5_file}")
        print(f"H5 exists: {os.path.exists(h5_file)}")
else:
    # Local execution: Go to project root
    project_dir = r'C:\Users\alexg\Downloads\CMU vPCF Project'
    os.chdir(project_dir)
    print(f"Changed to: {os.getcwd()}")
    
    h5_file = r'Experimentally-obtained vPCF Testing\data\vPCF_test_2.h5'
    dm3_file = r'Experimentally-obtained vPCF Testing\data\vPCF_test_2.dm3'
    print(f"H5 exists: {os.path.exists(h5_file)}")
    print(f"DM3 exists: {os.path.exists(dm3_file)}")

Running on VS Code Colab Extension
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted successfully at /content/drive
Changed to: /content/drive/My Drive/CMU vPCF Project


In [24]:
# For local execution, we need to add src/ from the project root
# The notebook is in Experimentally-obtained vPCF Testing/, so we go up one level
if not IN_COLAB:
    # Local: add src from project root
    src_path = os.path.join(os.getcwd(), 'src')
    if os.path.exists(src_path):
        sys.path.insert(0, src_path)
        print(f"Added to sys.path: {src_path}")
    else:
        print(f"Warning: src path not found at {src_path}")
else:
    # For Colab (VS Code or Web), we're in the project root now
    # vpcf_data_loader is in Experimentally-obtained vPCF Testing/
    vpcf_test_path = os.path.join(os.getcwd(), 'Experimentally-obtained vPCF Testing')
    src_path = os.path.join(os.getcwd(), 'src')
    
    if os.path.exists(vpcf_test_path):
        sys.path.insert(0, vpcf_test_path)
        print(f"Added to sys.path: {vpcf_test_path}")
    if os.path.exists(src_path):
        sys.path.insert(0, src_path)
        print(f"Added to sys.path: {src_path}")

# Verify imports
try:
    from vpcf_data_loader import load_vpcf_file, check_dependencies
    print("✓ Successfully imported vpcf_data_loader")
    
    # Check dependencies
    print("\nDependencies:")
    for name, available in check_dependencies().items():
        status = "✓ YES" if available else "✗ NO"
        print(f"  {name}: {status}")
except ImportError as e:
    print(f"✗ Import error: {e}")
    print(f"\nCurrent working directory: {os.getcwd()}")
    print(f"sys.path entries:")
    for path in sys.path[:5]:
        print(f"  {path}")
    print("\nNote: Make sure you're in the correct directory or have uploaded the necessary files")

Added to sys.path: /content/drive/My Drive/CMU vPCF Project/Experimentally-obtained vPCF Testing
✓ Successfully imported vpcf_data_loader

Dependencies:
  h5py: ✓ YES
  hyperspy: ✓ YES
  ncempy: ✗ NO


## Step 2: Load and Preprocess Data

In [25]:
import numpy as np
from vpcf_data_loader import load_vpcf_file, combine_datasets

# File paths are already set in the setup cell
print(f"H5 file exists: {os.path.exists(h5_file)}")
print(f"DM3 file exists: {os.path.exists(dm3_file)}")

H5 file exists: True
DM3 file exists: False


In [26]:
# Load H5 data with histogram features
# Use a smaller subset first for testing
print("Loading H5 data...")
h5_dataset = load_vpcf_file(
    h5_file,
    feature_method="histogram",  # "flatten", "histogram", "statistical", "combined"
    normalize="minmax",
    max_frames=500,  # Start with 500 frames for testing; remove for full dataset
    verbose=True
)

print(f"Loaded: {h5_dataset}")
print(f"Feature matrix shape: {h5_dataset.features.shape}")

Loading H5 data...
Loading 500 vPCF images from /content/drive/My Drive/CMU vPCF Project/Experimentally-obtained vPCF Testing/data/vPCF_test_2.h5...
  Loaded images with shape: (500, 999, 999)
Extracting features using method: histogram
  Feature shape: (500, 64)
Normalizing features using method: minmax
Loaded: VPCFDataset(n_samples=500, feature_dim=64, has_labels=False)
Feature matrix shape: (500, 64)


In [16]:
# Optionally load DM3 data
print("Loading DM3 data...")
dm3_dataset = load_vpcf_file(
    dm3_file,
    feature_method="histogram",
    normalize="minmax",
    verbose=True
)

print(f"Loaded: {dm3_dataset}")

Loading DM3 data...


NameError: name 'load_vpcf_file' is not defined

In [29]:
# Debug: Check available mount points and directories
import subprocess

print("Current working directory:", os.getcwd())
print("\nContents of /:")
for item in os.listdir('/'):
    if os.path.isdir(os.path.join('/', item)) and item not in ['proc', 'sys', 'dev']:
        print(f"  {item}/")

print("\nChecking for mounted Windows paths...")
for drive in ['C:', '/mnt/c', '/workspace', '/root']:
    if os.path.exists(drive):
        print(f"Found: {drive}")
        try:
            contents = os.listdir(drive)
            print(f"  Contents: {contents[:5]}")
        except:
            pass

# Try to find vPCF_test_2.h5
print("\nSearching for vPCF_test_2.h5...")
result = subprocess.run(['find', '/', '-name', 'vPCF_test_2.h5', '-type', 'f', '2>/dev/null'], 
                       capture_output=True, text=True, timeout=5)
if result.stdout:
    print("Found:")
    for line in result.stdout.strip().split('\n')[:5]:
        print(f"  {line}")
else:
    print("  Not found in accessible paths")

Current working directory: /content/drive/My Drive/CMU vPCF Project

Contents of /:
  libx32/
  tmp/
  usr/
  srv/
  boot/
  etc/
  lib64/
  media/
  mnt/
  root/
  lib/
  home/
  bin/
  opt/
  sbin/
  lib32/
  var/
  run/
  content/
  kaggle/
  tools/
  datalab/
  python-apt/

Checking for mounted Windows paths...
Found: /root
  Contents: ['.profile', '.bashrc', '.keras', '.cache', '.hyperspy']

Searching for vPCF_test_2.h5...
  Not found in accessible paths


## Step 3: Train DEC Model

In [33]:
# Verify model files exist in Experimentally-obtained vPCF Testing folder
vpcf_folder = os.path.join(os.getcwd(), 'Experimentally-obtained vPCF Testing')
print(f"VPCF folder: {vpcf_folder}")
print(f"Folder exists: {os.path.exists(vpcf_folder)}")

print("\nModel files:")
for fname in ['DEC.py', 'IDEC.py', 'metrics.py']:
    fpath = os.path.join(vpcf_folder, fname)
    exists = os.path.exists(fpath)
    size_mb = os.path.getsize(fpath) / 1024 if exists else 0
    print(f"  {fname}: {'✓' if exists else '✗'} ({size_mb:.1f} KB)")

VPCF folder: /content/drive/My Drive/CMU vPCF Project/Experimentally-obtained vPCF Testing
Folder exists: True

Model files:
  DEC.py: ✗ (0.0 KB)
  IDEC.py: ✗ (0.0 KB)
  metrics.py: ✗ (0.0 KB)


In [32]:
# Initialize DEC model
# Ensure the Experimentally-obtained vPCF Testing folder is in path
vpcf_folder = os.path.join(os.getcwd(), 'Experimentally-obtained vPCF Testing')
if vpcf_folder not in sys.path:
    sys.path.insert(0, vpcf_folder)

from DEC import DEC

# Prepare data
x = h5_dataset.features
n_clusters = 10
dims = [x.shape[1], 500, 500, 2000, 10]

# Create save directory
dec_save_dir = os.path.join(vpcf_folder, 'results', 'dec')
os.makedirs(dec_save_dir, exist_ok=True)

# Initialize DEC
dec = DEC(
    dims=dims,
    n_clusters=n_clusters,
    save_dir=dec_save_dir
)
print(f"✓ DEC model initialized")
print(f"  Clusters: {n_clusters}")
print(f"  Input features: {x.shape[1]}")
print(f"  Data shape: {x.shape}")

ModuleNotFoundError: No module named 'DEC'

In [ ]:
# Pretrain autoencoder and train clustering
print("Starting DEC training...")
print(f"  Epochs: 50 (pretraining)")
print(f"  Max iterations: 1000 (clustering)")

dec.compile(optimizer='sgd')
dec_labels = dec.fit(
    x,
    y=None,
    maxiter=1000,  # Increase to 8000+ for full training
    update_interval=140,
    batch_size=256
)

print(f"\n✓ DEC training complete!")
unique_labels = len(np.unique(dec_labels))
print(f"  Clusters found: {unique_labels}")
print(f"  Cluster distribution: {np.bincount(dec_labels)}")

## Step 4: Train IDEC Model

In [ ]:
# Initialize IDEC model
from IDEC import IDEC

# Create save directory
idec_save_dir = os.path.join(os.getcwd(), 'results', 'idec')
os.makedirs(idec_save_dir, exist_ok=True)

# Initialize IDEC
idec = IDEC(
    dims=dims,
    n_clusters=n_clusters,
    gamma=0.1,
    save_dir=idec_save_dir
)
print(f"✓ IDEC model initialized")
print(f"  Clusters: {n_clusters}")
print(f"  Input features: {x.shape[1]}")
print(f"  Data shape: {x.shape}")

In [ ]:
# Pretrain autoencoder
print("Pretraining autoencoder...")
idec.pretrain(x, epochs=50, batch_size=256)

In [ ]:
# Train IDEC model
print("Starting IDEC training...")
print(f"  Max iterations: 1000 (clustering)")

idec.compile(optimizer='sgd')
idec_labels = idec.fit(
    x,
    y=None,
    maxiter=1000,  # Increase to 8000+ for full training
    update_interval=140,
    batch_size=256
)

print(f"\n✓ IDEC training complete!")
unique_labels = len(np.unique(idec_labels))
print(f"  Clusters found: {unique_labels}")
print(f"  Cluster distribution: {np.bincount(idec_labels)}")

## Step 5: Evaluate Metrics

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

print("DEC Metrics:")
print(f"  Silhouette Score: {silhouette_score(x, dec_labels):.4f}")
print(f"  Davies-Bouldin Index: {davies_bouldin_score(x, dec_labels):.4f}")
print(f"  Calinski-Harabasz Index: {calinski_harabasz_score(x, dec_labels):.4f}")

print("\nIDEC Metrics:")
print(f"  Silhouette Score: {silhouette_score(x, idec_labels):.4f}")
print(f"  Davies-Bouldin Index: {davies_bouldin_score(x, idec_labels):.4f}")
print(f"  Calinski-Harabasz Index: {calinski_harabasz_score(x, idec_labels):.4f}")

## Step 6: Save Results

In [ ]:
import pandas as pd

# Save cluster assignments
results_df = pd.DataFrame({
    'sample_idx': np.arange(len(dec_labels)),
    'dec_cluster': dec_labels,
    'idec_cluster': idec_labels
})

results_df.to_csv('./Experimentally-obtained vPCF Testing/results/cluster_assignments.csv', index=False)
print("Saved cluster assignments")

# Cluster agreement
agreement = (dec_labels == idec_labels).mean() * 100
print(f"\nCluster agreement: {agreement:.2f}%")

## Step 7: Visualize Results

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# DEC cluster distribution
dec_counts = np.bincount(dec_labels, minlength=n_clusters)
axes[0].bar(range(n_clusters), dec_counts)
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Count')
axes[0].set_title('DEC Cluster Distribution')

# IDEC cluster distribution
idec_counts = np.bincount(idec_labels, minlength=n_clusters)
axes[1].bar(range(n_clusters), idec_counts)
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Count')
axes[1].set_title('IDEC Cluster Distribution')

plt.tight_layout()
plt.savefig('./Experimentally-obtained vPCF Testing/results/cluster_comparison.png', dpi=150)
plt.show()

print("Saved cluster comparison plot")

## Step 8: Download Results (if on Colab)

In [ ]:
if IN_COLAB:
    from google.colab import files
    
    results_dir = './Experimentally-obtained vPCF Testing/results'
    
    # Download all results
    print("Files to download:")
    for f in os.listdir(results_dir):
        print(f"  {f}")
    
    print("\nDownloading results...")
    # Zip and download
    import shutil
    shutil.make_archive('vpcf_results', 'zip', results_dir)
    files.download('vpcf_results.zip')
    print("Download complete!")
else:
    print("Results saved to: ./Experimentally-obtained vPCF Testing/results")